# Diagnose rinex file with Pride configurations
***
This notebook allows you to re-run a rinex file with various Pride configurations and compare the outputs.

## Import packages

In [1]:
from datetime import datetime
import logging

from es_sfgtools.processing.operations.gnss_ops import PridePdpConfig, rinex_to_kin, novatel_to_rinex


# Create a logger
logger = logging.getLogger('es_sfgtools')
logger.setLevel(logging.DEBUG)  # Set the logging level to DEBUG

# Create a stream handler to output logs to the notebook
stream_handler = logging.StreamHandler()
stream_handler.setLevel(logging.DEBUG)

# Create a formatter and set it for the handler
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
stream_handler.setFormatter(formatter)

# Add the handler to the logger
logger.addHandler(stream_handler)

***


## Generate or load rinex file

### 1. Set site name

In [2]:
site= 'ATW2'   # 4-char site name

### 2. (Optional) Load raw novatel file and convert to rinex

In [ ]:
novatel_file_path = ''          # path to novatel file
rinex_output_dir = '.'          # output directory path

novatel_to_rinex(source=novatel_file_path,
                 site=site,
                 write_dir=rinex_output_dir)

### 3. Set rinex file path

In [3]:
rinex_file_path = '/Users/terry/repos/seafloor_geodesy_notebooks/rinex_files/atw2001a.24o'    # path to rinex file

***

## Rinex to Kin with PridePPP

### 1. Set the configuration for the PRIDE PDP processing

In [26]:
# System -- GNSS System(s) to use. Default is "GREC23J" which is “GPS/GLONASS/Galileo/BDS/BDS-2/BDS-3/QZSS”
GNSS_system = "GREC23J"

# Frequency -- The GNSS frequencies to use. Default is "G12 R12 E15 C26 J12, 
# Refer to Table 5-4 in PRIDE-PPP-AR v.3.0 manual for more options.
# These must coincide with GNSS systems chosen
frequencies = ["G12", "R12", "E15", "C26", "J12"]

# Loose Edit -- disable strict editing mode, which should be used when high dynamic data quality is poor. Default is True.
loose_edit = True     # True or False

# Cut-off Elevation -- The elevation cutoff angle in degrees (0-60 degrees). Default is 7.
cutoff_elevation = 7    # 0-60 (int)

# Start and End times for processing, leave as None if you want to process the entire observation file.
start = None         # example datetime(2021, 1, 1, 0, 0, 0)
end = None           # example datetime(2021, 1, 1, 0, 0, 0)

# Processing interval, values range from 0.02s to 30s. If this item is not specified and the configuration file is specified, 
# the processing interval in the configuration file will be read, otherwise, the sampling rate of the observation file is used by default.
interval = None     # example 0.1 (float)

# Use 2nd ionospheric delay model with CODE's GIM product. When this option is not entered, no higher-order ionospheric correction is performed.
high_ion = False      # True or False

# Enter one or more of "S" "O" "P", e.g SO for solid, ocean, and polar tides. Default is "SOP", which uses all tides.
tides = "SOP"

# Local pdp3 binary path - if needed
local_pdp3_path = '/Users/terry/.PRIDE_PPPAR_BIN/pdp3'


2. Generate Pride pdp command using configs

In [27]:
if not rinex_file_path:
    raise ValueError("RINEX file path is required")

if not site:
    raise ValueError("Site name is required")

pride_object = PridePdpConfig(system=GNSS_system,
                              frequency=frequencies,
                              loose_edit=loose_edit,
                              cutoff_elevation=cutoff_elevation,
                              start=start,
                              end=end,
                              interval=interval,
                              high_ion=high_ion,
                              tides=tides,
                              local_pdp3_path=local_pdp3_path)

pdp_command = pride_object.generate_pdp_command(site=site, 
                                                local_file_path=rinex_file_path)

print("Generated PridePdp command:")
print(' '.join(pdp_command))


Generated PridePdp command:
/Users/terry/.PRIDE_PPPAR_BIN/pdp3 -m K --loose-edit --site ATW2 /Users/terry/repos/seafloor_geodesy_notebooks/rinex_files/atw2001a.24o


3. Run Pride with generate pdp command

In [30]:
# Set the kinematic file output directory path and PRIDE directory path
kin_output_dir = '/Users/terry/repos/seafloor_geodesy_notebooks/kin_files'        # kinematic file output directory path
pride_dir = '.'              # PRIDE directory path

# Convert RINEX to KIN
kin_file = rinex_to_kin(source=rinex_file_path,
                        writedir=kin_output_dir,
                        pridedir=pride_dir,
                        site=site,
                        PridePdpConfig=pride_object)

2024-10-04 17:16:39,421 - es_sfgtools.processing.operations.gnss_ops - INFO - Converting RINEX file /Users/terry/repos/seafloor_geodesy_notebooks/rinex_files/atw2001a.24o to kin file
2024-10-04 17:16:39,421 - es_sfgtools.processing.operations.gnss_ops - INFO - Running PDP3 command: /Users/terry/.PRIDE_PPPAR_BIN/pdp3 -m K --loose-edit --site ATW2 /Users/terry/repos/seafloor_geodesy_notebooks/rinex_files/atw2001a.24o
2024-10-04 17:16:39,498 - es_sfgtools.processing.operations.gnss_ops - ERROR - b"usage: dirname string [...]\n\x1berror:\x1b PRIDE PPP-AR configuration file doesn't exist: /config_template\n"


source: local_path=PosixPath('/Users/terry/repos/seafloor_geodesy_notebooks/rinex_files/atw2001a.24o') type=<AssetType.RINEX: 'rinex'> id=None network=None station=None survey=None timestamp_data_start=None timestamp_data_end=None timestamp_created=None parent_id=None size=None


IndexError: list index out of range

***
# Parse outputs